# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² Mass Spectrometry dataset using the `mlcroissant` library. You'll learn to load the Croissant-described dataset, inspect its record sets and fields, extract data into pandas DataFrames, and perform elementary data analysis—all referencing entities by their `@id`.

### Dataset Source
The dataset Croissant schema is provided via the following URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`. We use the schema URL as the entrypoint.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
List the available record sets (`cr:RecordSet`), each field (`cr:Field`), and the associated columns by their `@id`. These `@id`s will be used for all further references to data entities in this notebook.

In [ ]:
# List all record set, field, and column @ids
def list_metadata_overview(meta):
    print('--- Record Sets ---')
    for rs in meta.record_sets:
        print(f"RecordSet @id: {rs.id} / Name: {rs.name}")
        for fld in rs.fields:
            print(f"  Field @id: {fld.id} / Name: {fld.name} / DataType: {fld.data_type}")
            for col in fld.columns:
                print(f"    Column @id: {col.id} / Name: {col.name}")
        print()
    if not meta.record_sets:
        print('(No top-level record sets found in metadata. Checking file objects...)')
        if hasattr(meta, 'file_objects'):
            for file_obj in meta.file_objects:
                print(f"FileObject @id: {file_obj.id} / name: {getattr(file_obj, 'name', None)} / encoding: {getattr(file_obj, 'encoding', None)}")
                if hasattr(file_obj, 'record_set') and file_obj.record_set:
                    for rs in file_obj.record_set:
                        print(f"  RecordSet @id: {rs.id} / Name: {rs.name}")
                        for fld in rs.fields:
                            print(f"    Field @id: {fld.id} / Name: {fld.name} / DataType: {fld.data_type}")
                            for col in fld.columns:
                                print(f"      Column @id: {col.id} / Name: {col.name}")
                        print()

list_metadata_overview(metadata)

**Note:** If no top-level record sets are reported above, they might be referenced indirectly via file objects. The code will attempt to find all Croissant record sets and print their IDs, fields, and columns.

## 3. Data Extraction
Use the identified RecordSet `@id` to extract records and load into pandas DataFrames. All subsequent operations will also reference the Field or Column by their `@id`.

In [ ]:
# Collect all record sets' @ids for extraction
record_set_ids = []

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id for rs in metadata.record_sets]
else:
    # Try to find record_sets via file_object
    if hasattr(metadata, 'file_objects'):
        for fileobj in metadata.file_objects:
            if hasattr(fileobj, 'record_set') and fileobj.record_set:
                for rs in fileobj.record_set:
                    record_set_ids.append(rs.id)

assert record_set_ids, 'No record_set ids found in the metadata.'

# Show discovered record sets
print('Record sets with records:')
for idx, rsid in enumerate(record_set_ids):
    print(f"{idx}: {rsid}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record_set {record_set_id} with shape {df.shape}")
        else:
            print(f"No records found for record_set {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Inspect the columns of the first record set
first_record_set = record_set_ids[0]
if first_record_set in dataframes:
    print('First 10 columns:')
    print(dataframes[first_record_set].columns[:10].tolist())
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some exploratory analysis by selecting a numeric field (by `@id`) from one record set. We will filter, normalize and group by a categorical field if available. All fields/columns are referenced by their `@id`.

In [ ]:
# Select the main DataFrame
df_id = first_record_set
df = dataframes[df_id]

# Find a likely numeric field/column @id for analysis
numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
if not numeric_candidates:
    # Try to coerce numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]

# Fallback: Try common column names as numeric
for candidate in ['cr:field_coverage_percentage', 'cr:field_molecular_weight', 'cr:field_peptide_count', 'cr:field_abundance', 'cr:field_MW', 'cr:field_pi']:
    if candidate in df.columns:
        numeric_field_id = candidate
        break
else:
    numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]

print(f"Using numeric_field_id: {numeric_field_id}")

# Set a threshold (e.g. mean) and filter
numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = numeric_series.mean() if not np.isnan(numeric_series.mean()) else 0
filtered_df = df[numeric_series > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    numeric_series[filtered_df.index] - numeric_series.mean()
) / numeric_series.std() if numeric_series.std() else 0
print(f"Normalized {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a group field
group_field_id = None
for candidate in ['cr:field_accession', 'cr:field_sample_id', 'cr:field_protein_name', 'cr:field_modification']:
    if candidate in filtered_df.columns:
        group_field_id = candidate
        break
if not group_field_id:
    for col in filtered_df.columns:
        if filtered_df[col].nunique() < filtered_df.shape[0] // 2:
            group_field_id = col
            break

if group_field_id:
    grouped = filtered_df.groupby(group_field_id, dropna=False).mean(numeric_only=True)
    print(f"Grouped by {group_field_id} (top 5 rows):")
    display(grouped.head())
else:
    print('No categorical/group field with low cardinality found.')

## 5. Visualization
Visualize the distribution of the selected numeric field. For demonstration, we will plot a histogram of the filtered/normalized numeric values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
plt.xlabel(f"Normalized {numeric_field_id}")
plt.ylabel("Frequency")
plt.title(f"Distribution of normalized values for {numeric_field_id}")
plt.show()

# If a group_field_id is available, boxplot
if group_field_id and group_field_id in filtered_df.columns:
    # Only plot if few groups
    n_groups = filtered_df[group_field_id].nunique()
    if 2 <= n_groups <= 10:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.xticks(rotation=30)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the Mass Spectrometry dataset using the Croissant standard and the `mlcroissant` package in Python. All metadata navigation and data extraction referenced dataset entities by their `@id`, ensuring precise, standards-based access. Further analyses can include advanced modeling, biomarker discovery, or in-depth protein quantitation, leveraging the rich structure provided by the Croissant format.